# Data-Driven Multi-Objective Quantum Optimization

本 Notebook 展示 Figure 4（CGFM 对照）和 Figure 5（DDTS 多目标优化）的完整小规模流程。

> **阶段说明：** 当前保留原项目的 PyTorch FM + LBFGS 后端，MindSpore FM 暂未实现。

## 运行规模与注意事项

提交版默认使用 canonical `quick`。历史运行中 Figure 4 约需 19 小时 16 分，Figure 5
约需 5 小时 50 分，合计可能超过 25 小时。需要快速检查代码路径时，请在参数单元把
`RUN_SCALE` 临时改为 `"test"`。所有产物写入被 Git 忽略的 `outputs/`。

In [ ]:
from __future__ import annotations

import importlib.util
import json
import subprocess
import sys
from pathlib import Path

import torch
from IPython.display import Image
from IPython.display import display as display_image


def find_case_root() -> Path:
    """Locate the example root from Jupyter's current working directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "utils").is_dir() and (candidate / "main.ipynb").is_file():
            return candidate
    raise RuntimeError("Run this notebook from its example directory.")


CASE_ROOT = find_case_root()
case_root_path = str(CASE_ROOT)
if case_root_path not in sys.path:
    sys.path.insert(0, case_root_path)
print(f"Case root: {CASE_ROOT}")

## 环境检查

环境文件固定了原项目验证过的核心依赖，并增加 Jupyter 执行工具。此处只检查导入，
不下载数据或依赖。

In [ ]:
REQUIRED_MODULES = {
    "numpy": "NumPy",
    "scipy": "SciPy",
    "sklearn": "scikit-learn",
    "torch": "PyTorch",
    "optuna": "Optuna",
    "dimod": "dimod",
    "neal": "D-Wave Neal",
    "matplotlib": "Matplotlib",
}
missing = [label for module, label in REQUIRED_MODULES.items() if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(f"Missing dependencies: {', '.join(missing)}. Create the environment from environment.yml.")

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}; CUDA available: {torch.cuda.is_available()}")

## 统一运行参数

`quick` 是正式小规模验收配置，`test` 是端到端烟测配置。runner 始终使用 `--resume`，
因此同一配置中断后可以继续运行。修改规模或其他配置时应使用新的输出目录。

In [ ]:
RUN_SCALE = "quick"
RUN_FIGURE4 = True
RUN_FIGURE5 = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_ROOT = CASE_ROOT / "outputs"

if RUN_SCALE not in {"quick", "test"}:
    raise ValueError("RUN_SCALE must be 'quick' or 'test' in this notebook.")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Scale={RUN_SCALE}, device={DEVICE}, outputs={OUTPUT_ROOT}")

## 合金数据生成

四种次生相的归一化比例首先映射到总量为 0.2 的体积分数，再计算热导率、杨氏模量、
密度、热膨胀系数偏差和温度偏差。数据完全由固定 seed 本地合成，不依赖外部数据集。

In [ ]:
from utils.alloy_dataset_generator import generate_initial_dataset_multi_objective

preview_rows, preview_stats = generate_initial_dataset_multi_objective(num_samples=5, seed=0)
print(json.dumps(preview_stats, indent=2, ensure_ascii=False))
preview_rows[:2]

## FM、QUBO 与 CGFM

当前二阶 FM 使用 PyTorch + LBFGS 拟合离散设计与目标值。线性项和两两交互项随后转换为
QUBO；system 与 one-hot 约束按统一配置归一化并加权。Figure 4 比较直接四相编码
`wo_cgfm` 与三角度映射 `w_cgfm`，模拟退火从全部 reads 中选择能量最低的可行候选。

In [ ]:
def run_module(module: str, *arguments: str) -> None:
    """Run one supported CLI module and fail immediately on an error."""
    command = [sys.executable, "-m", module, *arguments]
    print("Running:", subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, cwd=CASE_ROOT, check=True)


def load_json(path: Path) -> dict[str, object]:
    """Load a generated JSON object."""
    with path.open(encoding="utf-8") as stream:
        payload = json.load(stream)
    if not isinstance(payload, dict):
        raise TypeError(f"Expected a JSON object in {path}")
    return payload

## Figure 4：CGFM 对照实验

对五个单目标分别比较直接编码和 CGFM 编码。实验结束后从 summary 生成聚合曲线；
canonical `quick` 还会运行严格验收器。

In [ ]:
FIGURE4_DIR = OUTPUT_ROOT / f"figure4_{RUN_SCALE}"
FIGURE4_SUMMARY = FIGURE4_DIR / "figure4_summary.json"
FIGURE4_PLOT = FIGURE4_DIR / "figure4.png"

if RUN_FIGURE4:
    run_module(
        "utils.figure4_runner",
        "--output-dir",
        str(FIGURE4_DIR),
        "--preset",
        RUN_SCALE,
        "--settings",
        "wo_cgfm",
        "w_cgfm",
        "--device",
        DEVICE,
        "--resume",
    )
    run_module(
        "utils.plot_figure4",
        "--summary",
        str(FIGURE4_SUMMARY),
        "--output",
        str(FIGURE4_PLOT),
    )
    if RUN_SCALE == "quick":
        run_module("utils.validate_reproduction", "figure4", "--summary", str(FIGURE4_SUMMARY))
    else:
        print("test scale completed; strict validation is intentionally quick-only.")

    figure4_summary = load_json(FIGURE4_SUMMARY)
    print("Figure 4 trajectories:", len(figure4_summary.get("trajectories", [])))
    display_image(Image(filename=str(FIGURE4_PLOT)))

## Figure 5：DDTS 多目标实验

比较 DDTS 人工目标 `w_ddts` 与 weighted-sum baseline `wo_ddts`，记录 proposed
solution、实际新增数据和 replacement，并从 summary 重建 Pareto front。

In [ ]:
FIGURE5_DIR = OUTPUT_ROOT / f"figure5_{RUN_SCALE}"
FIGURE5_SUMMARY = FIGURE5_DIR / "figure5_summary.json"
FIGURE5_PLOT = FIGURE5_DIR / "figure5.png"

if RUN_FIGURE5:
    run_module(
        "utils.figure5_runner",
        "--output-dir",
        str(FIGURE5_DIR),
        "--preset",
        RUN_SCALE,
        "--settings",
        "w_ddts",
        "wo_ddts",
        "--device",
        DEVICE,
        "--resume",
    )
    run_module(
        "utils.plot_figure5",
        "--summary",
        str(FIGURE5_SUMMARY),
        "--output",
        str(FIGURE5_PLOT),
    )
    if RUN_SCALE == "quick":
        run_module("utils.validate_reproduction", "figure5", "--summary", str(FIGURE5_SUMMARY))
    else:
        print("test scale completed; strict validation is intentionally quick-only.")

    figure5_summary = load_json(FIGURE5_SUMMARY)
    print("Figure 5 trajectories:", len(figure5_summary.get("trajectories", [])))
    display_image(Image(filename=str(FIGURE5_PLOT)))

## 输出与后续工作

运行产物位于 `outputs/figure4_<scale>/` 和 `outputs/figure5_<scale>/`，包括 manifest、
checkpoint、summary、validation、日志与图片，均不会进入 Git。后续工作是在保持本
Notebook 与 schema 接口不变的前提下，将 `utils/fm_torch.py` 替换为 MindSpore FM。

In [ ]:
for label, enabled, directory in (
    ("Figure 4", RUN_FIGURE4, FIGURE4_DIR),
    ("Figure 5", RUN_FIGURE5, FIGURE5_DIR),
):
    if enabled:
        generated = sorted(path.name for path in directory.glob("*"))
        print(f"{label}: {generated}")